In [2]:
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator
from datetime import datetime
import pandas as pd
import sqlite3

# -----------------------------
# Task 1: Ingest Data
# -----------------------------
def ingest_data():
    df = pd.read_csv('data/faculty.csv')
    df.to_csv('data/raw_faculty.csv', index=False)

# -----------------------------
# Task 2: Clean Data
# -----------------------------
def clean_data():
    df = pd.read_csv('data/raw_faculty.csv')
    df = df.dropna()
    df.to_csv('data/clean_faculty.csv', index=False)

# -----------------------------
# Task 3: Validate Data
# -----------------------------
def validate_data():
    df = pd.read_csv('data/clean_faculty.csv')
    assert df['faculty_id'].notnull().all()
    assert df['hours'].max() <= 40

# -----------------------------
# Task 4: Load to Database
# -----------------------------
def load_data():
    df = pd.read_csv('data/clean_faculty.csv')
    conn = sqlite3.connect('data/faculty.db')
    df.to_sql('faculty_workload', conn, if_exists='replace', index=False)
    conn.close()

# -----------------------------
# DAG Definition
# -----------------------------
default_args = {
    'start_date': datetime(2024, 1, 1),
}

with DAG(
    dag_id='faculty_workload_pipeline',
    schedule='@daily',   # ✅ FIXED
    catchup=False,
    default_args=default_args
) as dag:

    t1 = PythonOperator(
        task_id='ingest_data',
        python_callable=ingest_data
    )

    t2 = PythonOperator(
        task_id='clean_data',
        python_callable=clean_data
    )

    t3 = PythonOperator(
        task_id='validate_data',
        python_callable=validate_data
    )

    t4 = PythonOperator(
        task_id='load_data',
        python_callable=load_data
    )

    t1 >> t2 >> t3 >> t4